# Final Task B 17-class - ConvNeXt-Tiny

Train ConvNeXt-Tiny directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [ ]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


In [ ]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [ ]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "convnext_tiny"
DISPLAY_NAME = "ConvNeXt-Tiny"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


In [ ]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


In [ ]:
summarize_results(results)


## Continue after Vast interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. Then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


In [ ]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


In [ ]:
# Resume label-smoothing=0.0 from its saved checkpoint
run_remaining("taskB_label_smoothing_0.0", 42)


In [ ]:
# Remaining label-smoothing=0.2 ablation
run_remaining("taskB_label_smoothing_0.2", 42)


In [ ]:
results = list(continued_results.values())
summarize_results(results)
